# 04 · Evaluate on the ESCAPE benchmark test split

Loads the model from notebook 03 and evaluates it on the ESCAPE test split.
Self-contained.

> **Data:** point `TEST_URL` at your ESCAPE test CSV.

In [ ]:
# === Setup: install a transformers version with the classic BERT tokenizer ===
# (Colab's bundled transformers is too new and mishandles ProtBERT-BFD.)
%pip install -q transformers==4.46.3 "tokenizers>=0.20,<0.21" sentencepiece

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)   # expect 'cuda' — set Runtime > Change runtime type > GPU

## Config

In [ ]:
MODEL_NAME = 'Rostlab/prot_bert_bfd'
MAX_LEN    = 200

# TODO: set to your ESCAPE test split
TEST_URL   = 'https://raw.githubusercontent.com/BioGavin/AMP-BERT/main/data/raw/escape/escape_test.csv'
MODEL_DIR  = 'models/amp_bert_escape'   # from notebook 03
OUTPUT_DIR = 'results/escape_test'
PRED_CSV   = 'results/escape_test_predictions.csv'

In [ ]:
# === Core logic (self-contained; mirrors src/amp_bert/) ===
import pandas as pd, numpy as np
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             matthews_corrcoef, roc_auc_score)


def load_dataset(path, shuffle=False, random_state=0):
    df = pd.read_csv(path, index_col=0)
    return df.sample(frac=1, random_state=random_state) if shuffle else df


class AmpDataset(Dataset):
    """Tokenised view over an AMP DataFrame (columns: aa_seq, AMP)."""
    def __init__(self, df, tokenizer_name=MODEL_NAME, max_len=MAX_LEN):
        self.df = df.reset_index(drop=True)
        self.tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_name, do_lower_case=False, use_fast=False)
        self.max_len = max_len
        self.seqs = list(self.df['aa_seq'])
        self.labels = list(self.df['AMP'].astype(int))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        seq = ' '.join(''.join(self.seqs[idx].split()))
        ids = self.tokenizer(seq, truncation=True, padding='max_length',
                             max_length=self.max_len)
        sample = {k: torch.tensor(v) for k, v in ids.items()}
        sample['labels'] = torch.tensor(self.labels[idx])
        return sample


def compute_metrics(pred):
    labels = pred.label_ids
    logits = pred.predictions
    preds = logits.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='binary', zero_division=0)
    m = {'accuracy': accuracy_score(labels, preds), 'f1': f1,
         'precision': precision, 'recall': recall,
         'mcc': matthews_corrcoef(labels, preds)}
    if len(np.unique(labels)) == 2:
        z = logits - logits.max(axis=-1, keepdims=True)
        probs = np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)
        m['roc_auc'] = roc_auc_score(labels, probs[:, 1])
    return m

## Load data, model & evaluate

In [ ]:
test_df = load_dataset(TEST_URL)
print(test_df.shape)
test_dataset = AmpDataset(test_df)

In [ ]:
import os
assert os.path.exists(MODEL_DIR), f'No model at {MODEL_DIR}. Run notebook 03 first.'
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
args = TrainingArguments(output_dir=OUTPUT_DIR, per_device_eval_batch_size=56,
                        report_to='none')
trainer = Trainer(model=model, args=args, compute_metrics=compute_metrics)

In [ ]:
predictions, label_ids, metrics = trainer.predict(test_dataset)
metrics

In [ ]:
import os
os.makedirs('results', exist_ok=True)
out = test_df.copy()
out['pred'] = predictions.argmax(-1)
out.to_csv(PRED_CSV)
print('wrote', PRED_CSV)